In [1]:
!pip install google-cloud-bigquery-connection
!pip install google-cloud-bigquery

In [13]:
# Import necessary libraries
from google.cloud import bigquery
from google.api_core import exceptions


# --- Configuration ---
# Replace with your desired BigQuery dataset and table name
# The project ID 'qwiklabs-gcp-03-42d2ec04c144' is automatically used by the client
# if not explicitly specified, assuming the notebook is running in a GCP environment
# with appropriate credentials.
PROJECT_ID = "qwiklabs-gcp-03-42d2ec04c144"
DATASET_ID = "aurora_bay"  # e.g., "my_new_dataset"
TABLE_ID = "aurora_bay_faqs"      # e.g., "aurora_bay_faqs_table"
GCS_URI = "gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv"
LOCATION = "US"

# --- Initialize BigQuery client ---
client = bigquery.Client()

# --- Define table schema (important for CSV imports) ---
# You should inspect your CSV file to determine the correct column names and types.
# This is an example schema; adjust it based on the actual content of aurora-bay-faqs.csv.
schema = [
    bigquery.SchemaField("question", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("answer", "STRING", mode="NULLABLE"),
    # Add more fields as per your CSV structure
    # bigquery.SchemaField("column_name", "DATA_TYPE", mode="NULLABLE/REQUIRED"),
]

# --- Configure the load job ---
job_config = bigquery.LoadJobConfig(
    schema=schema,
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,  # Skip the header row in the CSV
    autodetect=False,     # Set to True if you want BigQuery to infer schema,
                          # but defining it explicitly is often safer.
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Overwrite table if it exists
    # write_disposition=bigquery.WriteDisposition.WRITE_APPEND, # Append to table if it exists
    # write_disposition=bigquery.WriteDisposition.WRITE_EMPTY,  # Fail if table exists
)

# --- Construct a full dataset reference ---
dataset_ref = client.dataset(DATASET_ID, )

# --- Create the dataset if it doesn't exist (optional, but good practice) ---
try:
    client.get_dataset(dataset_ref)
    print(f"Dataset '{DATASET_ID}' already exists.")
except Exception:
    print(f"Dataset '{DATASET_ID}' not found. Creating it...")
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = LOCATION  # Specify your desired location, e.g., "US", "EU"
    client.create_dataset(dataset)
    print(f"Dataset '{DATASET_ID}' created.")

# --- Start the load job ---
print(f"Starting job to load '{GCS_URI}' into '{DATASET_ID}.{TABLE_ID}'...")
load_job = client.load_table_from_uri(
    GCS_URI,
    dataset_ref.table(TABLE_ID),
    job_config=job_config,
)

# --- Wait for the job to complete ---
load_job.result()  # Waits for the job to finish
print("Job finished.")

# --- Verify the load ---
destination_table = client.get_table(dataset_ref.table(TABLE_ID))
print(f"Loaded {destination_table.num_rows} rows into {DATASET_ID}.{TABLE_ID}.")











Dataset 'aurora_bay' not found. Creating it...
Dataset 'aurora_bay' created.
Starting job to load 'gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv' into 'aurora_bay.aurora_bay_faqs'...
Job finished.
Loaded 50 rows into aurora_bay.aurora_bay_faqs.


In [14]:
!bq mk --connection --connection_type=CLOUD_RESOURCE --location=us --project_id={PROJECT_ID} "embedding_conn"
!bq show --location=us --connection --project_id={PROJECT_ID} "embedding_conn"

BigQuery error in mk operation: Already Exists: Connection
projects/157254411328/locations/us/connections/embedding_conn
Connection qwiklabs-gcp-03-42d2ec04c144.us.embedding_conn

               name                friendlyName   description    Last modified         type        hasCredential                                            properties                                            
 -------------------------------- -------------- ------------- ----------------- ---------------- --------------- ----------------------------------------------------------------------------------------------- 
  157254411328.us.embedding_conn                                05 Dec 15:08:17   CLOUD_RESOURCE   False           {"serviceAccountId": "bqcx-157254411328-70zg@gcp-sa-bigquery-condel.iam.gserviceaccount.com"}  



In [19]:
# Update you service acccount here
connection_service_account = "bqcx-157254411328-70zg@gcp-sa-bigquery-condel.iam.gserviceaccount.com"
connection_member = f"serviceAccount:{connection_service_account}"

!gcloud projects add-iam-policy-binding {PROJECT_ID} --member={connection_member} --role='roles/aiplatform.user' --condition=None --quiet

Updated IAM policy for project [qwiklabs-gcp-03-42d2ec04c144].
bindings:
- members:
  - serviceAccount:service-157254411328@gcp-sa-vertex-nb.iam.gserviceaccount.com
  role: roles/aiplatform.colabServiceAgent
- members:
  - serviceAccount:service-157254411328@gcp-sa-aiplatform-vm.iam.gserviceaccount.com
  role: roles/aiplatform.notebookServiceAgent
- members:
  - serviceAccount:service-157254411328@gcp-sa-aiplatform.iam.gserviceaccount.com
  role: roles/aiplatform.serviceAgent
- members:
  - serviceAccount:bqcx-157254411328-70zg@gcp-sa-bigquery-condel.iam.gserviceaccount.com
  role: roles/aiplatform.user
- members:
  - serviceAccount:qwiklabs-gcp-03-42d2ec04c144@qwiklabs-gcp-03-42d2ec04c144.iam.gserviceaccount.com
  role: roles/bigquery.admin
- members:
  - serviceAccount:service-157254411328@gcp-sa-cloudaicompanion.iam.gserviceaccount.com
  role: roles/cloudaicompanion.serviceAgent
- members:
  - serviceAccount:157254411328@cloudbuild.gserviceaccount.com
  role: roles/cloudbuild.builds

In [20]:
%%bigquery --project qwiklabs-gcp-03-42d2ec04c144
CREATE OR REPLACE MODEL `aurora_bay.embedding_model` REMOTE WITH CONNECTION `us.embedding_conn` OPTIONS (ENDPOINT = 'text-embedding-005');

Query is running:   0%|          |

""


# Task
Generate embeddings for the `question` and `answer` columns from the `aurora_bay.aurora_bay_faqs` table using the `aurora_bay.embedding_model` and store them in a new table named `aurora_bay.aurora_bay_faqs_with_embeddings` along with the original `question` and `answer` fields.

## Generate embeddings

### Subtask:
Generate embeddings for the `question` and `answer` columns from the `aurora_bay.aurora_bay_faqs` table using the `aurora_bay.embedding_model`.


**Reasoning**:
Generate embeddings for the 'question' and 'answer' columns from the `aurora_bay.aurora_bay_faqs` table using the `aurora_bay.embedding_model`.



In [22]:
%%bigquery --project qwiklabs-gcp-03-42d2ec04c144
CREATE OR REPLACE TABLE `aurora_bay.aurora_bay_faq_embeddings` AS SELECT * FROM ML.GENERATE_EMBEDDING(MODEL `qwiklabs-gcp-03-42d2ec04c144.aurora_bay.embedding_model`, (SELECT question, answer, CONCAT(question, ' ', answer) AS content FROM `qwiklabs-gcp-03-42d2ec04c144.aurora_bay.aurora_bay_faqs`), STRUCT(TRUE AS flatten_json_output));

Query is running:   0%|          |

""


In [24]:
# prompt: Program a chatbot that can use a vector search to find the data required from the aurora_bay_faq_embeddings table to answer the user’s question.

from google.cloud import bigquery
import pandas as pd

# Initialize BigQuery client
client = bigquery.Client()

def get_embedding(text, model_name="text-embedding-005"):
    """Generates embeddings for a given text using a specified model."""
    query = f"""
    SELECT
      ml_generate_embedding_result
    FROM
      ML.GENERATE_EMBEDDING(MODEL `qwiklabs-gcp-03-42d2ec04c144.aurora_bay.embedding_model`,
        (SELECT "{text}" AS content))
    """
    query_job = client.query(query)
    results = query_job.result()
    for row in results:
        return row.ml_generate_embedding_result[0]
    return None

def find_similar_faqs(user_question, top_n=3):
    """
    Finds the most similar FAQs to the user's question using vector search.

    Args:
        user_question (str): The question asked by the user.
        top_n (int): The number of top similar FAQs to retrieve.

    Returns:
        pandas.DataFrame: A DataFrame containing the top N similar FAQs.
    """
    # Get the embedding for the user's question
    question_embedding = get_embedding(user_question)

    if not question_embedding:
        return pd.DataFrame()

    # Construct the query to find similar FAQs using vector search
    # We'll use the dot product for similarity, assuming embeddings are normalized.
    # If not normalized, cosine similarity would be more appropriate.
    # For simplicity here, we'll use dot product and order by it.
    # The `aurora_bay_faq_embeddings` table is assumed to have a column named `ml_generate_embedding_result`
    # which contains the embeddings.
    query = f"""
    SELECT
        question,
        answer,
        VECTOR_COSINE_DISTANCE(embedding, @question_embedding) AS similarity
    FROM
        `qwiklabs-gcp-03-42d2ec04c144.aurora_bay.aurora_bay_faq_embeddings`
    ORDER BY
        similarity DESC
    LIMIT {top_n}
    """

    # Define query parameters
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("question_embedding", "ARRAY<FLOAT64>", values=question_embedding),
        ]
    )

    # Execute the query
    query_job = client.query(query, job_config=job_config)
    results = query_job.to_dataframe()

    return results

def chatbot_response(user_question):
    """
    Generates a chatbot response based on the user's question using vector search.

    Args:
        user_question (str): The question asked by the user.

    Returns:
        str: The chatbot's response.
    """
    similar_faqs = find_similar_faqs(user_question)

    if similar_faqs.empty:
        return "I'm sorry, I couldn't find any relevant information to answer your question."
    else:
        # For simplicity, we'll just return the answer from the most similar FAQ.
        # In a more advanced chatbot, you might combine answers or ask clarifying questions.
        best_match = similar_faqs.iloc[0]
        return f"Based on your question, here's what I found:\n\nQ: {best_match['question']}\nA: {best_match['answer']}"

# --- Example Usage ---
if __name__ == "__main__":
    while True:
        user_input = input("You: ")
        if user_input.lower() in ["quit", "exit", "bye"]:
            print("Chatbot: Goodbye!")
            break
        response = chatbot_response(user_input)
        print(f"Chatbot: {response}")

You: culture


TypeError: ScalarQueryParameter.__init__() got an unexpected keyword argument 'values'